# Mood-Aware Book recommendation engine
This notebook handles the integration of semantic text embeddings and numerical metadata into a **Qdrant Vector Database**.

**Key Workflow**
1. Combine Sentence embeddings with scaled metadata
2. Connect to Qdrant and define the collection schema
3. Efficiently upload thousands of Points (Vectors + Payloads)
4. Optimise the collection for filtered searches. 

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

import pandas as pd
import numpy as np
import joblib
import os

from sklearn.preprocessing import StandardScaler


## Load Processed Data
We are importing two primary components: 
1. **The metadata:** A CSV file containing book titles, authors, and engineered features. 
2. **The embeddings:** A pre-computed NumPy array representing the semantic meaning of book descriptions. 

In [2]:
df = pd.read_csv("../data/processed/books_with_features.csv")

embeddings = np.load("../data/processed/description_embeddings.npy")

print("Loaded dataset:", df.shape)
print("Loaded embeddings:", embeddings.shape)

Loaded dataset: (5006, 36)
Loaded embeddings: (5006, 384)


## Feature Scaling
To build a "Hybrid" search, we concatenate our high-dimensional embeddings with specific numerical metadata. 

Since raw metadata (like word counts) has a different scale than normalised embeddings, we use `StandardScaler` to bring them into a comparable range. We save this scaler to ensure we can transform future user queries the same way. 

In [3]:
from sklearn.preprocessing import StandardScaler

metadata_cols = [
    "description_word_count",
    "num_tags",
    "average_rating"
]

scaler = StandardScaler()

metadata_matrix = scaler.fit_transform(df[metadata_cols])

combined_vectors = np.hstack([
    embeddings,
    metadata_matrix
])

### Combine Embeddings and Metadata
We now stack the embeddings with the scaled metadata to create our final input vectors. 

In [4]:
metadata_cols = [
    "description_word_count",
    "num_tags",
    "average_rating"
]

scaler = StandardScaler()

metadata_matrix = scaler.fit_transform(df[metadata_cols])

print("Metadata matrix shape:", metadata_matrix.shape)


# Save scaler for reuse later

os.makedirs("../artifacts", exist_ok=True)

joblib.dump(
    scaler,
    "../artifacts/metadata_scaler.pkl"
)

print("Metadata scaler saved")



Metadata matrix shape: (5006, 3)
Metadata scaler saved


## Prepare Vector Points
In Qdrant, a Point consists of:
1. **ID:** A unique identifier. 
2. **Vector:** The 387-dimensional numerical representation. 
3. **Payload:** Metadata (Title, Moods, Tags) that we can use for filtering results later. 

In [5]:
combined_vectors = np.hstack([
    embeddings,
    metadata_matrix
])

print("Combined vector shape:", combined_vectors.shape)

Combined vector shape: (5006, 387)


Create payload objects

In [6]:
points = [
    PointStruct(
        id=int(idx),
        vector=combined_vectors[idx].tolist(),
        payload={
            "title": df.iloc[idx]["title"],
            "authors": df.iloc[idx]["authors"],
            "tags": df.iloc[idx]["tags_cleaned"],
            "average_rating": float(df.iloc[idx]["average_rating"]),
            "moods": [
                col.replace("mood_", "")
                for col in df.columns
                if col.startswith("mood_")
                and df.iloc[idx][col] == 1
            ]
        }
    )
    for idx in range(len(df))
]

print(f"Created {len(points)} vector points")

Created 5006 vector points


## Database setup
We connect to the local Qdrant instance. We need to ensure that the docker container is running. 
TODO: Automate this :)

In [7]:
client = QdrantClient(
    host="localhost",
    port=6333
)

print("Connected to Qdrant")

Connected to Qdrant


### Collection Initialisation
We define the collection using **Cosine Distance**. This is standard for NLP tasks as it measures the angle between vectors rather than their absolute magnitude.

In [8]:
# Force delete if exists, then create fresh
collection_name = "book_recommender"

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)
    print("Old collection deleted")

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=combined_vectors.shape[1],
        distance=Distance.COSINE
    )
)
print("Fresh collection created")

Old collection deleted
Fresh collection created


## Batch Ingestion
Uploading 5000+ points in a single request can be unstable. We split the data into batches of 500 for a more reliable upload process. 

In [9]:
batch_size = 500

for i in range(0, len(points), batch_size):

    batch = points[i:i + batch_size]

    client.upsert(
        collection_name=collection_name,
        points=batch
    )

    print(f"Uploaded {i} → {min(i + batch_size, len(points))}")


print("Upload complete")

Uploaded 0 → 500
Uploaded 500 → 1000
Uploaded 1000 → 1500
Uploaded 1500 → 2000
Uploaded 2000 → 2500
Uploaded 2500 → 3000
Uploaded 3000 → 3500
Uploaded 3500 → 4000
Uploaded 4000 → 4500
Uploaded 4500 → 5000
Uploaded 5000 → 5006
Upload complete


## Verification and indexing
First, we confirm taht all points were successfully stored and that the cluster status is Green.

In [10]:
collection_info = client.get_collection(collection_name)

print("\nCollection Status:", collection_info.status)
print("Total Points Stored:", collection_info.points_count)


Collection Status: green
Total Points Stored: 5006


### Optimising Search with Payload Indexes
To make filtered searches quickly, we create specialised indexes on our payload fields.

In [11]:
client.create_payload_index(
    collection_name=collection_name,
    field_name="average_rating",
    field_schema="float"
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="tags",
    field_schema="keyword"
)

print("Payload indexes created")

Payload indexes created


### Retrieve sample book
Finally, we retrieve a sample point by ID to ensure the payload was stored correctly and the moods were parsed as expected. 

In [12]:
sample_point = client.retrieve(
    collection_name=collection_name,
    ids=[0],
    with_payload=True
)

if sample_point:

    print("\nSample Book from Qdrant:")
    print("Title:", sample_point[0].payload["title"])
    print("Rating:", sample_point[0].payload["average_rating"])
    print("Moods:", sample_point[0].payload["moods"])

else:

    print("No sample book retrieved")


Sample Book from Qdrant:
Title: The Unschooled Wizard (Sun Wolf and Starhawk, #1-2)
Rating: 4.03
Moods: ['escapist']
